# 🔬 Auditoria Computacional: Conjectura de Stănică–Maitra para HRSBF Cúbicas
## Resolução Incondicional para todas as Dimensões Pares $n \not\equiv 0 \pmod{32}$

Este notebook executa a **auditoria computacional completa e reproduzível** do artigo científico e do dossiê de auditoria.

### O que é verificado aqui:
1. **Teorema de Ward em $t=16$**: Todas as **136.697 condições de divisibilidade** sobre as 43 geradoras (35 cúbicas, 7 quadráticas e a forma linear $L$) nos 4.080 representantes de órbitas completas.
2. **Matrizes de Transferência Analíticas**: Cálculo exato de $W_H(0)$ e prova de que em $t=32$ o resto mod 2048 é $512 \ne 0$ (fronteira exata de Ward).
3. **Espaços Reduzidos $t \in \{2, 4, 8\}$**: Avaliação de todo o span com a forma linear $L$ e ausência de valores bent.
4. **Buscas Exaustivas de Baixa Dimensão**: Confirmação por FWHT em $n \le 12$ cúbicas e $n=8, 10$ quárticas.
5. **Verificador Standalone**: Implementação alternativa 100% em biblioteca padrão de Python (zero dependências externas).


In [ ]:
import itertools, math, random, sys, time, collections
import numpy as np
print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")


---
### Etapa 1: Teorema de Ward em $t = 16$ (136.697 Condições de Polarização de Pesos)
Audita a congruência de Ward nas 4 ordens:
* Ordem 1: $\binom{43}{1} = 43$ geradores, $\mathrm{wt}(w_i) \equiv 0 \pmod{16}$
* Ordem 2: $\binom{43}{2} = 903$ pares, $\mathrm{wt}(w_i \cap w_j) \equiv 0 \pmod 8$
* Ordem 3: $\binom{43}{3} = 12.341$ trios, $\mathrm{wt}(w_i \cap w_j \cap w_l) \equiv 0 \pmod 4$
* Ordem 4: $\binom{43}{4} = 123.410$ quartetos, $\mathrm{wt}(w_i \cap w_j \cap w_l \cap w_p) \equiv 0 \pmod 2$


In [ ]:
def audit_ward_t16():
    t0 = time.time()
    t = 16
    N = 1 << t
    xs = np.arange(N, dtype=np.uint32)
    rots = np.stack([((xs << s) | (xs >> (t - s))) & (N - 1) for s in range(t)])
    full = rots[8] != xs
    reps = np.unique(rots.min(0)[full])
    print(f"Representantes de orbitas completas em F_2^16 \\ V_8: {len(reps)}")
    assert len(reps) == 4080

    # 35 composicoes ciclicas canonicas de lacunas (g1, g2, g3) com soma 16
    comps = sorted({min(g[i:] + g[:i] for i in range(3)) 
                    for g in itertools.product(range(1, t), repeat=3) if sum(g) == t})
    assert len(comps) == 35

    cub_orbs = []
    for a, b, c in comps:
        mon = (0, a, a + b)
        o = {tuple(sorted((i + s) % t for i in mon)) for s in range(t)}
        cub_orbs.append(sorted(o))

    # 7 orbitas quadraticas completas d=1..7
    quad_orbs = [sorted({tuple(sorted((i, (i + d) % t))) for i in range(t)}) for d in range(1, 8)]
    assert len(quad_orbs) == 7

    # Gerar truth tables vetorizadas
    def truth(orb):
        v = np.zeros(N, dtype=np.uint8)
        for mon in orb:
            m = sum(1 << i for i in mon)
            v ^= ((xs & m) == m).astype(np.uint8)
        return v

    tabs = [truth(o) for o in cub_orbs + quad_orbs]
    L = np.zeros(N, dtype=np.uint8)
    for i in range(t):
        L ^= ((xs >> i) & 1).astype(np.uint8)
    tabs_L = tabs + [L]

    # Converter para inteiros binarios de 4080 bits sobre os representantes
    def to_int(v):
        return int.from_bytes(np.packbits(v, bitorder="little").tobytes(), "little")

    b = [to_int(tb[reps]) for tb in tabs_L]
    k = len(b)
    assert k == 43

    f = [0, 0, 0, 0]
    for i in range(k):
        f[0] += b[i].bit_count() % 16 != 0
    for i, j in itertools.combinations(range(k), 2):
        f[1] += (b[i] & b[j]).bit_count() % 8 != 0
    for i, j, l in itertools.combinations(range(k), 3):
        f[2] += (b[i] & b[j] & b[l]).bit_count() % 4 != 0
    for i, j in itertools.combinations(range(k), 2):
        bij = b[i] & b[j]
        for l in range(j + 1, k):
            bijl = bij & b[l]
            for p in range(l + 1, k):
                f[3] += (bijl & b[p]).bit_count() % 2 != 0

    total_tested = 43 + 903 + 12341 + 123410
    print(f"Total de subconjuntos auditados (Ordens 1 a 4): {total_tested}")
    print(f"Falhas encontradas por ordem de Ward: {f}")
    assert f == [0, 0, 0, 0], "Violacao de Ward detectada!"
    print(f"==> SUCESSO: Todas as 136.697 condicoes satisfeitas em {time.time()-t0:.2f} segundos!")
    print("==> Para toda funcao g no span: wt(g) = 0 (mod 256) ==> W_g(0) = 0 (mod 512) != +-256 (BENT IMPOSSIVEL)")

audit_ward_t16()


---
### Etapa 2: Matriz de Transferência Exata e Limite Estrutural em $t = 32$
Calcula $W_H(0) = \mathrm{tr}(M^{32})$ para a órbita elementar $H(x) = \sum_{i=0}^{31} x_i x_{i+1} x_{i+2}$.


In [ ]:
def transfer_matrix_check():
    states = list(itertools.product(range(2), repeat=2))
    matrix = [[0] * 4 for _ in states]
    for i, (a, b) in enumerate(states):
        for j, (b2, c) in enumerate(states):
            if b == b2:
                matrix[i][j] = (-1) ** (a * b * c)

    def multiply(a, b):
        return [[sum(a[i][k] * b[k][j] for k in range(4)) for j in range(4)] for i in range(4)]

    def trace_power(n):
        p = [[int(i == j) for j in range(4)] for i in range(4)]
        for _ in range(n):
            p = multiply(p, matrix)
        return sum(p[i][i] for i in range(4))

    # Validacao para n=3..10 contra soma direta
    for n in range(3, 11):
        direct = sum((-1) ** sum(x[i] * x[(i + 1) % n] * x[(i + 2) % n] for i in range(n))
                     for x in itertools.product(range(2), repeat=n))
        assert direct == trace_power(n)
    print("Matriz de transferencia 4x4 validada contra somas exaustivas para n=3..10.")

    # Em t=32:
    walsh32 = trace_power(32)
    weight32 = ((1 << 32) - walsh32) // 2
    orbit_weight = weight32 // 32
    rem2048 = orbit_weight % 2048

    print(f"t=32: W_H(0) = {walsh32}, wt(H) = {weight32}")
    print(f"wt(H)/32 = {orbit_weight} = 512 (mod 2048) -> resto: {rem2048}")
    assert walsh32 == 85032960
    assert weight32 == 2104967168
    assert rem2048 == 512
    print("==> Confirmado: O contraexemplo H(x) tem resto 512 != 0 mod 2048.")
    print("==> Isso prova que t=16 e a fronteira exata da divisibilidade direta de Ward!")

transfer_matrix_check()


---
### Etapa 3: Espaços Reduzidos $t \in \{2, 4, 8\}$ com Forma Linear $L$
Testa exaustivamente todas as funções em $\mathcal{F}_t$ e confirma que $\pm 2^{t/2}$ nunca ocorre no espectro de Walsh.


In [ ]:
def orbits(n, deg):
    seen, out = set(), []
    for mon in itertools.combinations(range(n), deg):
        if mon in seen: continue
        o = {tuple(sorted((i + s) % n for i in mon)) for s in range(n)}
        seen |= o; out.append(sorted(o))
    return out

for t in (2, 4, 8):
    gens = orbits(t, 3) + [o for o in orbits(t, 2) if len(o) == t] + [[(i,) for i in range(t)]]
    N = 1 << t
    xs = np.arange(N, dtype=np.uint32)
    tbs = []
    for o in gens:
        v = np.zeros(N, dtype=np.uint8)
        for mon in o:
            m = sum(1 << i for i in mon)
            v ^= ((xs & m) == m).astype(np.uint8)
        tbs.append(v)
    vals = set()
    for c in range(1 << len(tbs)):
        v = np.zeros(N, dtype=np.uint8)
        for i in range(len(tbs)):
            if (c >> i) & 1: v ^= tbs[i]
        vals.add(N - 2 * int(v.sum()))
    target = 1 << (t // 2)
    absent = (target not in vals) and (-target not in vals)
    print(f"t={t:2d}: {len(tbs):2d} geradoras, 2^{len(tbs)} funcoes; alvo bent +-{target} ausente: {absent}")
    assert absent
    if t == 8:
        assert all(v % 32 == 0 for v in vals)


---
### Etapa 4: Buscas Exaustivas de Baixa Dimensão ($n \le 12$ cúbicas e $n=8, 10$ quárticas)
Validação por transformada rápida de Walsh-Hadamard (FWHT) de todas as candidatas com peso bent compatível.


In [ ]:
def fwht(a):
    a = a.astype(np.int64).copy(); h = 1
    while h < len(a):
        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2
    return a

def is_bent(v, n):
    return bool(np.all(np.abs(fwht(1 - 2 * v.astype(np.int64))) == (1 << (n // 2))))

def brute(n, deg):
    orbs = orbits(n, deg); k = len(orbs)
    N = 1 << n
    xs = np.arange(N, dtype=np.uint32)
    tts = []
    for o in orbs:
        v = np.zeros(N, dtype=np.uint8)
        for mon in o:
            m = sum(1 << i for i in mon)
            v ^= ((xs & m) == m).astype(np.uint8)
        tts.append(v)
    ints = [int.from_bytes(np.packbits(x, bitorder="little").tobytes(), "little") for x in tts]
    target = {N // 2 - (1 << (n // 2 - 1)), N // 2 + (1 << (n // 2 - 1))}
    cur = cand = bent = 0
    for g in range(1, 1 << k):
        cur ^= ints[(g & -g).bit_length() - 1]
        if cur.bit_count() in target:
            cand += 1
            gray = g ^ (g >> 1)
            v = np.zeros(N, dtype=np.uint8)
            for i in range(k):
                if (gray >> i) & 1: v ^= tts[i]
            bent += is_bent(v, n)
    return k, cand, bent

jobs = [(4, 3), (6, 3), (8, 3), (10, 3), (12, 3), (8, 4), (10, 4)]
for n, d in jobs:
    t0 = time.time()
    k, cand, bent = brute(n, d)
    print(f"Grau {d}, n={n:2d}: {k:2d} orbitas, 2^{k:2d} funcoes, candidatas de peso: {cand:6d}, BENT: {bent} ({time.time()-t0:.1f}s)")
    assert bent == 0


---
## 🎉 Conclusão da Auditoria

Todas as verificações finitas terminaram com **zero falhas e zero contraexemplos**.
* O Teorema de Redução de Ordem Ímpar (Lema 3.1) garante que se existisse uma HRSBF bent em $n$ variáveis, sua restrição induziria uma função bent em $t \in \{2, 4, 8, 16\}$.
* As verificações acima demonstram analiticamente e computacionalmente que nenhuma dessas funções pode ser bent.
* Portanto, **não existe nenhuma função bent homogênea simétrica por rotação cúbica em qualquer dimensão par $n \not\equiv 0 \pmod{32}$**.
